# LOCA-PRAM — Batch analysis across every assay

Runs the downsampled real-image inference path over every folder under
`assays/`, using every image in every cycle. Per-assay outputs land in
`assays/<assay_name>/analysis/`:

* `results.csv` — one row per image at the default (threshold, nms_kernel).
* `example_detections/` — 6 example images with detections X-marked, chosen
  to span the density range uniformly (nearest to `linspace(min, max, 6)`
  by detection count).
* `boxplot_over_time.png` — box-and-whisker of detection counts per cycle.
* `per_tile_over_time.png` — one thin line per tile position parsed from
  `tile_<row>_<col>_*.jpg` filenames, with the mean-across-tiles as a bold
  overlay. Falls back to ordinal-within-cycle if filenames don't encode a
  position.

Cycle folders may be named `cycle_XXXX` or `demo_XXXX`; both are picked up
and their number is used as the time index (cycle N → `(N-1) * MINUTES_PER_CYCLE`).

**Pipeline** — matches `LOCA_PRAM_negative_eval.ipynb` exactly (same
DOWNSCALE_FACTOR, MARKER_THRESHOLD, marker buffer, edge margin,
`normalize_tile`, `compute_forbidden_mask_fast`, and NMS detector). Only
the batch driver and analysis outputs differ.


## 1. Imports

In [1]:
import os
import re
import glob
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import cv2
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

Image.MAX_IMAGE_PIXELS = None


C:\Users\hankeun2\Desktop\brian_test\pram_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Model architecture

Self-contained copy of `GaussianMixtureModel` (matches
`LOCA_PRAM_negative_eval.ipynb`).

In [2]:
class conv_block(nn.Module):
    def __init__(self, in_channels, out_channels, dropout=0.1, norm_groups=6, dilation=1):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, 1, dilation, dilation=dilation, bias=True),
            nn.GroupNorm(norm_groups, out_channels),
            nn.ELU(),
            nn.Conv2d(out_channels, out_channels, 3, 1, dilation, dilation=dilation, bias=True),
            nn.GroupNorm(norm_groups, out_channels),
            nn.ELU(),
        )
    def forward(self, x): return self.conv(x)


class up_conv(nn.Module):
    def __init__(self, in_channels, out_channels, dropout=0.1, norm_groups=6):
        super().__init__()
        self.up = nn.Sequential(
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True),
            nn.Conv2d(in_channels, out_channels, 3, 1, padding="same", bias=True),
        )
    def forward(self, x): return self.up(x)


class multi_head(nn.Module):
    def __init__(self, in_channels, out_channels, dropout=0.1, norm_groups=6):
        super().__init__()
        self.multi_head = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, 3, 1, padding="same", bias=True),
            nn.GroupNorm(norm_groups, in_channels),
            nn.ELU(),
            nn.Conv2d(in_channels, out_channels, 1, 1, padding="same", bias=True),
        )
    def forward(self, x): return self.multi_head(x)


class GaussianMixtureModel(nn.Module):
    def __init__(self, num_channels):
        super().__init__()
        self.out_channels_heads = (1, 3, 3, 1)
        self.num_channels = num_channels

        self.Conv1 = conv_block(num_channels, 36, norm_groups=6)
        self.Maxpool1 = nn.MaxPool2d(2, 2)
        self.Conv2 = conv_block(36, 72, norm_groups=6)
        self.Maxpool2 = nn.MaxPool2d(2, 2)
        self.Conv3 = conv_block(72, 144, norm_groups=6)
        self.Maxpool3 = nn.MaxPool2d(2, 2)
        self.Conv4 = nn.Sequential(
            conv_block(144, 288, norm_groups=6, dilation=2),
            conv_block(288, 288, norm_groups=6, dilation=4),
        )
        self.Up3 = up_conv(288, 144, norm_groups=6)
        self.Up_conv3 = conv_block(288, 144, norm_groups=6)
        self.Up2 = up_conv(144, 72, norm_groups=6)
        self.Up_conv2 = conv_block(144, 72, norm_groups=6)
        self.Up1 = up_conv(72, 36, norm_groups=6)
        self.Up_conv1 = conv_block(72, 36, norm_groups=6)
        self.dropout = nn.Dropout2d(p=0.3)
        self.sigma_eps = 0.001
        self.tanh_scale = 1.0
        self.mt_heads = nn.ModuleList([
            multi_head(36, ch, norm_groups=6) for ch in self.out_channels_heads
        ])
        self.initialize_weights()
        nn.init.constant_(self.mt_heads[0].multi_head[-1].bias, -8.1)
        nn.init.zeros_(self.mt_heads[0].multi_head[-1].weight)

    def initialize_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.Linear)):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
                if m.bias is not None: nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.GroupNorm):
                nn.init.constant_(m.weight, 1); nn.init.constant_(m.bias, 0)

    def forward(self, x, training=False):
        x1 = self.Conv1(x)
        x2 = self.Conv2(self.Maxpool1(x1))
        x3 = self.Conv3(self.Maxpool2(x2))
        x4 = self.Conv4(self.Maxpool3(x3))
        d3 = self.Up3(x4); d3 = torch.cat((x3, d3), dim=1); d3 = self.Up_conv3(d3)
        d2 = self.Up2(d3); d2 = torch.cat((x2, d2), dim=1); d2 = self.Up_conv2(d2)
        d1 = self.Up1(d2); d1 = torch.cat((x1, d1), dim=1); d1 = self.Up_conv1(d1)
        d = [h(d1) for h in self.mt_heads]
        p = torch.sigmoid(torch.clamp(d[0], min=-20.0, max=20.0))
        pxyn_mean = d[1]
        pxyn_mean[:, [0, 1], ...] = torch.tanh(pxyn_mean[:, [0, 1], ...]) * self.tanh_scale
        pxyn_mean[:, [2], ...] = torch.sigmoid(pxyn_mean[:, [2], ...]) * 100
        pxy_std = torch.sigmoid(d[2]) * 1.4 + 0.1
        bg = d[3]
        return p, pxyn_mean, pxy_std, bg


## 3. Config

In [3]:
# --- pipeline scales (must match negative_eval / downsampled inference) ---
DOWNSCALE_FACTOR = 2
BASE_WINDOW_SIZE = (256, 256)
window_size = (BASE_WINDOW_SIZE[0] // DOWNSCALE_FACTOR,
                BASE_WINDOW_SIZE[1] // DOWNSCALE_FACTOR)        # (128, 128)
NATIVE_WINDOW = (window_size[0] * DOWNSCALE_FACTOR,
                  window_size[1] * DOWNSCALE_FACTOR)             # (256, 256)

MARKER_THRESHOLD = 15

BASE_SIGMA_X_LOC = 9.79
BASE_SIGMA_Y_LOC = 7.88
BASE_MARGIN      = 12
SIGMA_X_LOC = BASE_SIGMA_X_LOC / DOWNSCALE_FACTOR
SIGMA_Y_LOC = BASE_SIGMA_Y_LOC / DOWNSCALE_FACTOR
MARGIN      = max(1, int(round(BASE_MARGIN / DOWNSCALE_FACTOR)))
BUFFER_SIGMA_MULT = 5.0
BUFFER_PX = int(round(BUFFER_SIGMA_MULT * max(SIGMA_X_LOC, SIGMA_Y_LOC)))
EDGE_MARGIN_TEST = 6

MIN_AREA           = 1
DEFAULT_THRESHOLD  = 0.006
DEFAULT_NMS_KERNEL = 7
OVERLAP_NATIVE      = 2 * EDGE_MARGIN_TEST * DOWNSCALE_FACTOR
DEDUP_RADIUS_NATIVE = 3

# --- batch I/O ---
ASSAYS_ROOT     = "assays"
ASSAY_GLOB      = "*"                    # every assay under assays/
DEMO_GLOBS      = ("demo_*", "cycle_*")  # accept both naming schemes
IMAGE_EXT       = ".jpg"
ANALYSIS_SUBDIR = "analysis"

MODEL_PATH        = "pram_dense_final_v7.pth"
MINUTES_PER_CYCLE = 5                    # cycle N -> (N-1) * this
N_EXAMPLE_IMAGES  = 6                    # spanning density range uniformly

print(f"window_size        = {window_size}")
print(f"NATIVE_WINDOW      = {NATIVE_WINDOW}")
print(f"BUFFER_PX          = {BUFFER_PX}, MARGIN = {MARGIN}, "
      f"EDGE_MARGIN_TEST = {EDGE_MARGIN_TEST}")
print(f"DEFAULT_THRESHOLD  = {DEFAULT_THRESHOLD}")
print(f"DEFAULT_NMS_KERNEL = {DEFAULT_NMS_KERNEL}")
print(f"ASSAYS_ROOT        = {ASSAYS_ROOT}")
print(f"MINUTES_PER_CYCLE  = {MINUTES_PER_CYCLE}")


window_size        = (128, 128)
NATIVE_WINDOW      = (256, 256)
BUFFER_PX          = 24, MARGIN = 6, EDGE_MARGIN_TEST = 6
DEFAULT_THRESHOLD  = 0.006
DEFAULT_NMS_KERNEL = 7
ASSAYS_ROOT        = assays
MINUTES_PER_CYCLE  = 5


## 4. Inference helpers

In [4]:
def load_real_native(path):
    arr = np.asarray(Image.open(path)).astype(np.float32)
    if arr.ndim == 3:
        arr = arr[:, :, 0]
    return arr


def downsample_image(img, factor=DOWNSCALE_FACTOR):
    if factor == 1:
        return img
    h, w = img.shape[:2]
    return cv2.resize(img, (w // factor, h // factor),
                      interpolation=cv2.INTER_AREA)


def normalize_tile(image):
    valid = image > MARKER_THRESHOLD
    if valid.any():
        m, s = image[valid].mean(), image[valid].std()
    else:
        m, s = image.mean(), image.std()
    if s == 0:
        s = 1
    return (image - m) / s


def compute_forbidden_mask_fast(image, buffer_px=BUFFER_PX,
                                 threshold=MARKER_THRESHOLD,
                                 margin=EDGE_MARGIN_TEST):
    H, W = image.shape
    non_marker = (image > threshold).astype(np.uint8)
    if non_marker.all():
        forbidden = np.zeros((H, W), dtype=bool)
    else:
        dist = cv2.distanceTransform(non_marker, cv2.DIST_L2, 3)
        forbidden = dist <= buffer_px
    if margin > 0:
        forbidden[:margin, :]  = True
        forbidden[-margin:, :] = True
        forbidden[:, :margin]  = True
        forbidden[:, -margin:] = True
    return forbidden


def detect_via_nms_xy(p_map_np, mu_map_np, forbidden_mask, p_threshold,
                       nms_kernel=5, refine_subpixel=True):
    H, W = p_map_np.shape
    above = (p_map_np > p_threshold) & ~forbidden_mask
    if not above.any():
        return np.array([]), np.array([])
    kernel = np.ones((nms_kernel, nms_kernel), dtype=np.uint8)
    p_max = cv2.dilate(p_map_np.astype(np.float32), kernel)
    peaks = above & (p_map_np >= p_max - 1e-9)
    yi, xi = np.where(peaks)
    if refine_subpixel:
        rx = xi.astype(np.float64) + mu_map_np[0, yi, xi]
        ry = yi.astype(np.float64) + mu_map_np[1, yi, xi]
    else:
        rx = xi.astype(np.float64); ry = yi.astype(np.float64)
    return rx, ry


def tile_starts(total, tile_size, stride):
    if total <= tile_size:
        return [0]
    starts = list(range(0, total - tile_size + 1, stride))
    if starts[-1] + tile_size < total:
        starts.append(total - tile_size)
    return starts


def dedup_positions(positions, radius=DEDUP_RADIUS_NATIVE):
    if len(positions) <= 1:
        return np.asarray(positions)
    from scipy.spatial import cKDTree
    pts = np.asarray(positions)
    tree = cKDTree(pts)
    pairs = tree.query_pairs(radius)
    drop = set()
    for i_, j_ in sorted(pairs):
        if i_ in drop or j_ in drop:
            continue
        drop.add(j_)
    keep = [i_ for i_ in range(len(pts)) if i_ not in drop]
    return pts[keep]


## 5. Per-image inference

`process_image` returns just the detection count for the batch pass;
`detect_for_viz` returns positions for the example-image renderer.

In [5]:
def process_image(model, device, path,
                  threshold=DEFAULT_THRESHOLD,
                  nms_kernel=DEFAULT_NMS_KERNEL,
                  overlap_native=OVERLAP_NATIVE,
                  dedup_radius=DEDUP_RADIUS_NATIVE):
    img = load_real_native(path)
    h, w = img.shape
    stride_y = NATIVE_WINDOW[0] - overlap_native
    stride_x = NATIVE_WINDOW[1] - overlap_native
    starts_y = tile_starts(h, NATIVE_WINDOW[0], stride_y)
    starts_x = tile_starts(w, NATIVE_WINDOW[1], stride_x)

    positions = []
    total_pixels = 0
    masked_pixels = 0
    max_p = 0.0

    with torch.no_grad():
        for iy in starts_y:
            for ix in starts_x:
                native_tile = img[iy:iy+NATIVE_WINDOW[0],
                                  ix:ix+NATIVE_WINDOW[1]]
                raw       = downsample_image(native_tile)
                norm      = normalize_tile(raw)
                forbidden = compute_forbidden_mask_fast(raw)

                x_in = torch.from_numpy(norm).float().unsqueeze(0).unsqueeze(0).to(device)
                p, pxy_mean, _, _ = model(x_in, training=False)
                p_map  = p[0, 0].cpu().numpy()
                mu_map = pxy_mean[0, :2].cpu().numpy()

                max_p = max(max_p, float(p_map.max()))
                total_pixels  += forbidden.size
                masked_pixels += int(forbidden.sum())

                cx128, cy128 = detect_via_nms_xy(
                    p_map, mu_map, forbidden, threshold, nms_kernel=nms_kernel)
                if len(cx128):
                    cx_native = cx128 * DOWNSCALE_FACTOR + ix
                    cy_native = cy128 * DOWNSCALE_FACTOR + iy
                    positions.append(np.stack([cx_native, cy_native], axis=1))

    if positions:
        deduped = dedup_positions(np.concatenate(positions, axis=0),
                                   radius=dedup_radius)
        n = len(deduped)
    else:
        n = 0

    return {
        "n_detections": n,
        "max_p":        max_p,
        "n_tiles":      len(starts_y) * len(starts_x),
        "mask_coverage": masked_pixels / total_pixels if total_pixels else 0.0,
        "image_h":      h,
        "image_w":      w,
    }


def detect_for_viz(model, device, path,
                    p_threshold=DEFAULT_THRESHOLD,
                    nms_kernel=DEFAULT_NMS_KERNEL,
                    overlap_native=OVERLAP_NATIVE,
                    dedup_radius=DEDUP_RADIUS_NATIVE):
    img = load_real_native(path)
    h, w = img.shape
    stride_y = NATIVE_WINDOW[0] - overlap_native
    stride_x = NATIVE_WINDOW[1] - overlap_native
    starts_y = tile_starts(h, NATIVE_WINDOW[0], stride_y)
    starts_x = tile_starts(w, NATIVE_WINDOW[1], stride_x)

    all_x, all_y = [], []
    with torch.no_grad():
        for iy in starts_y:
            for ix in starts_x:
                native_tile = img[iy:iy+NATIVE_WINDOW[0],
                                  ix:ix+NATIVE_WINDOW[1]]
                raw       = downsample_image(native_tile)
                norm      = normalize_tile(raw)
                forbidden = compute_forbidden_mask_fast(raw)
                x_in = torch.from_numpy(norm).float().unsqueeze(0).unsqueeze(0).to(device)
                p, pxy_mean, _, _ = model(x_in, training=False)
                p_map  = p[0, 0].cpu().numpy()
                mu_map = pxy_mean[0, :2].cpu().numpy()
                cx128, cy128 = detect_via_nms_xy(
                    p_map, mu_map, forbidden, p_threshold, nms_kernel=nms_kernel)
                if len(cx128):
                    all_x.append(cx128 * DOWNSCALE_FACTOR + ix)
                    all_y.append(cy128 * DOWNSCALE_FACTOR + iy)

    if not all_x:
        return img, np.array([]), np.array([])
    pts = np.stack([np.concatenate(all_x), np.concatenate(all_y)], axis=1)
    deduped = dedup_positions(pts, radius=dedup_radius)
    return img, deduped[:, 0], deduped[:, 1]


## 6. Load model

In [6]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = GaussianMixtureModel(num_channels=1)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.to(device).eval()
(model._orig_mod if hasattr(model, '_orig_mod') else model).tanh_scale = 1.0
print(f"Loaded {MODEL_PATH} on {device}")


Loaded pram_dense_final_v7.pth on cuda:0


## 7. Filename parsers

In [7]:
_TILE_RE = re.compile(r"tile_(\d+)_(\d+)_", re.IGNORECASE)
_CYCLE_RE = re.compile(r"(?:demo|cycle)_(\d+)", re.IGNORECASE)


def parse_tile_pos(filename):
    """Extract (row, col) from `tile_<r>_<c>_*.jpg`. None if no match."""
    m = _TILE_RE.match(os.path.basename(filename))
    return (int(m.group(1)), int(m.group(2))) if m else None


def parse_cycle_num(demo_name):
    """Extract cycle number from `demo_XXXX` or `cycle_XXXX`. None if no match."""
    m = _CYCLE_RE.match(os.path.basename(demo_name))
    return int(m.group(1)) if m else None


## 8. Per-assay analysis writers

Each function reads a single-assay dataframe (one row per image at the
default threshold) and writes one output file/dir. `save_example_detections`
picks 6 images whose counts are nearest to `linspace(min, max, 6)` so the
range is covered uniformly rather than clustered at the tail.

In [8]:
def save_example_detections(model, device, assay_df, out_dir,
                              n_examples=N_EXAMPLE_IMAGES,
                              marker_px=24, marker_lw=2):
    """6 example images with X-marked detections, spanning density range."""
    os.makedirs(out_dir, exist_ok=True)
    if len(assay_df) == 0:
        return []
    ranked = assay_df.sort_values("n_detections").reset_index(drop=True)
    n_pick = min(n_examples, len(ranked))
    lo, hi = int(ranked["n_detections"].min()), int(ranked["n_detections"].max())
    targets = np.linspace(lo, hi, n_pick)

    picked = []
    used = set()
    for t in targets:
        remaining = ranked[~ranked.index.isin(used)]
        if len(remaining) == 0:
            break
        idx = int((remaining["n_detections"] - t).abs().idxmin())
        used.add(idx)
        picked.append(idx)

    saved = []
    for k, idx in enumerate(picked):
        row = ranked.loc[idx]
        try:
            img, cx, cy = detect_for_viz(model, device, row["path"])
        except Exception as e:
            print(f"    skip example {row['image']}: {e}")
            continue
        p_lo, p_hi = np.percentile(img, [1, 99])
        norm = np.clip((img - p_lo) / (p_hi - p_lo + 1e-9), 0, 1)
        bgr = cv2.cvtColor((norm * 255).astype(np.uint8), cv2.COLOR_GRAY2BGR)
        for x, y in zip(cx, cy):
            cv2.drawMarker(bgr, (int(round(x)), int(round(y))),
                           color=(0, 0, 255),
                           markerType=cv2.MARKER_TILTED_CROSS,
                           markerSize=marker_px, thickness=marker_lw)
        safe = row["image"].replace("/", "_").replace(" ", "_")
        base = f"ex{k+1:02d}_n{int(row['n_detections'])}_{row['demo']}_{safe}"
        if not base.lower().endswith((".png", ".jpg", ".jpeg")):
            base += ".png"
        out = os.path.join(out_dir, base)
        cv2.imwrite(out, bgr)
        saved.append(out)
    return saved


def save_boxplot_over_time(assay_df, out_path,
                             minutes_per_cycle=MINUTES_PER_CYCLE):
    """One box per cycle (whiskers = 1.5*IQR, mean line marked)."""
    df = assay_df.copy()
    df["cycle"] = df["demo"].apply(parse_cycle_num)
    df = df.dropna(subset=["cycle"])
    if len(df) == 0:
        print(f"    no parseable cycles for boxplot -> {out_path}")
        return
    df["cycle"] = df["cycle"].astype(int)
    df["t_min"] = (df["cycle"] - 1) * minutes_per_cycle

    by_t = df.groupby("t_min")["n_detections"].apply(list)
    ts = sorted(by_t.index)
    data = [by_t[t] for t in ts]

    width_px = max(minutes_per_cycle * 0.6, 0.5)
    fig, ax = plt.subplots(figsize=(max(8, 0.35 * len(ts) + 4), 5))
    ax.boxplot(data, positions=ts, widths=width_px,
               showmeans=True, meanline=True)
    ax.set_xlabel("time (min, first cycle = t=0)")
    ax.set_ylabel(f"# detections per image "
                  f"(thr={DEFAULT_THRESHOLD}, nms={DEFAULT_NMS_KERNEL})")
    ax.set_title("Detections over time — box & whisker")
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(out_path, dpi=120, bbox_inches="tight")
    plt.close(fig)


def save_per_tile_over_time(assay_df, out_path,
                              minutes_per_cycle=MINUTES_PER_CYCLE):
    """One thin line per tile position, mean-across-tiles overlaid bold.

    Position parsed from `tile_<row>_<col>_*.jpg` when present. If no
    filename in the assay matches, falls back to ordinal-within-cycle
    (files sorted by name; ord_0, ord_1, ...).

    Duplicates — when the same tile has multiple captures at the same
    cycle, they're collapsed to their mean before plotting, so per-tile
    lines don't jag vertically at that timepoint and the mean-across-tiles
    isn't biased toward tiles that were re-imaged.
    """
    df = assay_df.copy()
    df["cycle"] = df["demo"].apply(parse_cycle_num)
    df = df.dropna(subset=["cycle"])
    if len(df) == 0:
        print(f"    no parseable cycles for per-tile plot -> {out_path}")
        return
    df["cycle"] = df["cycle"].astype(int)
    df["t_min"] = (df["cycle"] - 1) * minutes_per_cycle

    df["_pos"] = df["image"].apply(parse_tile_pos)
    matched = df["_pos"].notna().sum()
    if matched > 0:
        df["tile"] = df["_pos"].apply(
            lambda p: f"{p[0]},{p[1]}" if p is not None else None)
        df = df.dropna(subset=["tile"])
        tile_source = f"parsed from filename (matched {matched}/{len(df)})"
    else:
        df = df.sort_values(["cycle", "image"]).reset_index(drop=True)
        df["tile"] = "ord_" + df.groupby("cycle").cumcount().astype(str)
        tile_source = "ordinal within cycle (no `tile_r_c_` filenames found)"

    # Collapse duplicate captures at the same (tile, cycle) to their mean.
    agg = (df.groupby(["tile", "t_min"], as_index=False)["n_detections"]
             .mean())
    tiles = sorted(agg["tile"].unique())
    fig, ax = plt.subplots(figsize=(10, 5))
    for t in tiles:
        g = agg[agg["tile"] == t].sort_values("t_min")
        ax.plot(g["t_min"], g["n_detections"], "-", alpha=0.35,
                lw=0.9, color="steelblue")
    mean_line = agg.groupby("t_min")["n_detections"].mean().sort_index()
    ax.plot(mean_line.index, mean_line.values, "-o", color="crimson",
            lw=2.5, markersize=5, label="mean across tiles")
    ax.set_xlabel("time (min, first cycle = t=0)")
    ax.set_ylabel(f"# detections per image "
                  f"(thr={DEFAULT_THRESHOLD}, nms={DEFAULT_NMS_KERNEL})")
    ax.set_title(f"Per-tile detections over time ({len(tiles)} tiles; {tile_source})")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(out_path, dpi=120, bbox_inches="tight")
    plt.close(fig)


## 9. Discover assays

Every folder under `assays/` matching `ASSAY_GLOB` is a separate assay.
Within each, both `demo_*` and `cycle_*` cycle folders are picked up.

In [9]:
assay_dirs = sorted(d for d in glob.glob(os.path.join(ASSAYS_ROOT, ASSAY_GLOB))
                    if os.path.isdir(d))
print(f"Found {len(assay_dirs)} assays under {ASSAYS_ROOT}/:")

manifest = []
for d in assay_dirs:
    demo_dirs = sorted({p for pat in DEMO_GLOBS
                          for p in glob.glob(os.path.join(d, pat))
                          if os.path.isdir(p)})
    n_imgs = sum(len(glob.glob(os.path.join(dd, f"*{IMAGE_EXT}")))
                 for dd in demo_dirs)
    manifest.append((d, len(demo_dirs), n_imgs))
    print(f"  {os.path.basename(d)}: {len(demo_dirs)} cycle folders, {n_imgs} images")
print(f"\nTotal images to process: {sum(m[2] for m in manifest)}")


Found 7 assays under assays/:
  2026.06.22_neg_01: 12 cycle folders, 500 images
  2026.06.23_neg_02: 13 cycle folders, 325 images
  2026.06.24_pos_1pM_01_(code_had_bug_some_time_points_missing): 11 cycle folders, 275 images
  2026.06.25_neg_04_failed_capture_stock_used__r13_c13: 13 cycle folders, 325 images
  2026.07.13_19.15.39_blocking_test_without_capture_and_with_probeprotector_water_0.4OD_aunp: 1 cycle folders, 25 images
  2026.07.13_21.21.10_blocking_test_without_capture_and_without_probeprotector_water_0.4OD_aunp_1h_blocking_incubation: 13 cycle folders, 325 images
  2026.07.14_using_pdms_well_for_chemistry_followed_by_90min_blocking: 7 cycle folders, 10 images

Total images to process: 1785


## 10. Batch run

For every assay: run inference over every image at the default
(threshold, nms_kernel), write `results.csv` incrementally, then produce
the three analysis outputs. Per-assay outputs land in
`assays/<name>/analysis/`.

In [10]:
t_all = time.time()
overall_summary = []

for assay_dir in tqdm(assay_dirs, desc="assays"):
    assay_name = os.path.basename(assay_dir)
    out_dir = os.path.join(assay_dir, ANALYSIS_SUBDIR)
    os.makedirs(out_dir, exist_ok=True)
    results_csv = os.path.join(out_dir, "results.csv")

    demo_dirs = sorted({p for pat in DEMO_GLOBS
                          for p in glob.glob(os.path.join(assay_dir, pat))
                          if os.path.isdir(p)})
    if not demo_dirs:
        print(f"  {assay_name}: no cycle folders, skipping")
        continue

    rows = []
    t_assay = time.time()
    for demo in demo_dirs:
        demo_name = os.path.basename(demo)
        imgs = sorted(glob.glob(os.path.join(demo, f"*{IMAGE_EXT}")))
        for p in tqdm(imgs, desc=f"  {demo_name}", leave=False):
            try:
                out = process_image(model, device, p)
            except Exception as e:
                print(f"    !! {p}: {e}")
                continue
            rows.append({
                "assay":        assay_name,
                "demo":         demo_name,
                "image":        os.path.basename(p),
                "path":         p,
                "threshold":    DEFAULT_THRESHOLD,
                "nms_kernel":   DEFAULT_NMS_KERNEL,
                "n_detections": out["n_detections"],
                "max_p":        out["max_p"],
                "n_tiles":      out["n_tiles"],
                "mask_coverage": out["mask_coverage"],
                "image_h":      out["image_h"],
                "image_w":      out["image_w"],
            })
            if len(rows) % 20 == 0:
                pd.DataFrame(rows).to_csv(results_csv, index=False)

    assay_df = pd.DataFrame(rows)
    assay_df.to_csv(results_csv, index=False)
    dt = time.time() - t_assay
    print(f"  {assay_name}: {len(assay_df)} images in {dt:.1f}s -> {results_csv}")

    if len(assay_df) == 0:
        continue

    saved = save_example_detections(model, device, assay_df,
                                     os.path.join(out_dir, "example_detections"))
    print(f"    example_detections/: {len(saved)} images")

    save_boxplot_over_time(assay_df,
                             os.path.join(out_dir, "boxplot_over_time.png"))
    print(f"    boxplot_over_time.png")

    save_per_tile_over_time(assay_df,
                              os.path.join(out_dir, "per_tile_over_time.png"))
    print(f"    per_tile_over_time.png")

    overall_summary.append({
        "assay":  assay_name,
        "n_images": len(assay_df),
        "mean_n": float(assay_df["n_detections"].mean()),
        "median_n": float(assay_df["n_detections"].median()),
        "max_n":  int(assay_df["n_detections"].max()),
    })

print(f"\nAll assays done in {time.time()-t_all:.1f}s")
summary_df = pd.DataFrame(overall_summary)
summary_df


  demo_0001: 100%|████████████████████████████████████████████████████████████████████████████████████████| 25/25 [00:33<00:00,  1.30s/it]
                                                                                                                                          
  demo_0002: 100%|██████████████████████████████████████████████████████████████████████████████████████| 225/225 [04:59<00:00,  1.38s/it]
                                                                                                                                          
  demo_0003: 100%|████████████████████████████████████████████████████████████████████████████████████████| 25/25 [00:35<00:00,  1.39s/it]
                                                                                                                                          
  demo_0004: 100%|████████████████████████████████████████████████████████████████████████████████████████| 25/25 [00:34<00:00,  1.41s/it]
                           

  2026.06.22_neg_01: 500 images in 692.0s -> assays\2026.06.22_neg_01\analysis\results.csv
    example_detections/: 6 images
    boxplot_over_time.png


assays:  14%|█████████████▏                                                                              | 1/7 [11:42<1:10:15, 702.52s/it]

    per_tile_over_time.png



  demo_0001: 100%|████████████████████████████████████████████████████████████████████████████████████████| 25/25 [00:36<00:00,  1.36s/it]
                                                                                                                                          
  demo_0002: 100%|████████████████████████████████████████████████████████████████████████████████████████| 25/25 [00:35<00:00,  1.48s/it]
                                                                                                                                          
  demo_0003: 100%|████████████████████████████████████████████████████████████████████████████████████████| 25/25 [00:38<00:00,  1.64s/it]
                                                                                                                                          
  demo_0004: 100%|████████████████████████████████████████████████████████████████████████████████████████| 25/25 [00:39<00:00,  1.52s/it]
                          

...


  cycle_0001: 100%|█████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:01<00:00,  1.57s/it]
                                                                                                                                          
  cycle_0002: 100%|█████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:01<00:00,  1.48s/it]
                                                                                                                                          
  cycle_0003: 100%|█████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:01<00:00,  1.46s/it]
                                                                                                                                          
  cycle_0004: 100%|█████████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:04<00:00,  1.44s/it]
                          

  2026.07.14_using_pdms_well_for_chemistry_followed_by_90min_blocking: 10 images in 14.6s -> assays\2026.07.14_using_pdms_well_for_chemistry_followed_by_90min_blocking\analysis\results.csv
    example_detections/: 6 images
    boxplot_over_time.png


assays: 100%|██████████████████████████████████████████████████████████████████████████████████████████████| 7/7 [44:06<00:00, 378.12s/it]

    per_tile_over_time.png

All assays done in 2646.8s


,assay,n_images,mean_n,median_n,max_n
0,2026.06.22_neg_01,500,92.854000,75.5,1891
1,2026.06.23_neg_02,325,127.873846,92.0,861
2,2026.06.24_pos_1pM_01_(code_had_bug_some_time_...,275,904.763636,809.0,2656
3,2026.06.25_neg_04_failed_capture_stock_used__r...,325,624.661538,607.0,1770
4,2026.07.13_19.15.39_blocking_test_without_capt...,25,161.440000,142.0,421
5,2026.07.13_21.21.10_blocking_test_without_capt...,325,384.221538,375.0,1076
6,2026.07.14_using_pdms_well_for_chemistry_follo...,10,271.000000,259.0,471


## 11. (Optional) Quick look at any assay's outputs

Set `PEEK_ASSAY` below to inspect a specific assay's saved plots inline.
Leave it as-is to just peek at the first one found.

In [ ]:
PEEK_ASSAY = None   # e.g. "2026.07.14_using_pdms_well_..."; None = first assay

pick = None
if PEEK_ASSAY:
    cand = os.path.join(ASSAYS_ROOT, PEEK_ASSAY, ANALYSIS_SUBDIR)
    if os.path.isdir(cand):
        pick = cand
if pick is None:
    for d in assay_dirs:
        cand = os.path.join(d, ANALYSIS_SUBDIR)
        if os.path.isdir(cand):
            pick = cand
            break

if pick is None:
    print("no analysis outputs found yet")
else:
    print(f"peeking at {pick}")
    for name in ("boxplot_over_time.png", "per_tile_over_time.png"):
        p = os.path.join(pick, name)
        if os.path.isfile(p):
            img = plt.imread(p)
            fig, ax = plt.subplots(figsize=(10, 5))
            ax.imshow(img); ax.set_axis_off(); ax.set_title(name)
            plt.show()
    ex_dir = os.path.join(pick, "example_detections")
    if os.path.isdir(ex_dir):
        exs = sorted(glob.glob(os.path.join(ex_dir, "*.png")))
        print(f"{len(exs)} example detection images in {ex_dir}")
